In [ ]:
%%writefile app.py
import streamlit as st
import torch
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM

# 1. Konfigurasi Halaman Web
st.set_page_config(page_title="Qwen Fitness Expert", page_icon="🏋️‍♂️", layout="centered")
st.title("🏋️‍♂️ Qwen Fitness Expert Chatbot")
st.caption("Asisten kebugaran cerdas bertenaga Qwen2.5-1.5B (LoRA Fine-tuned)")

# 2. Fungsi Memuat Model (Di-cache agar tidak load berulang)
@st.cache_resource
def load_model():
    # PATH ASLI KEMBALI DIMASUKKAN KE SINI
    path_qwen = "/kaggle/input/models/muhammadzakikurnia/gwen/pytorch/default/1/qwen_fitness_lora_final"

    # Jika folder LoRA tidak punya tokenizer, pinjam dari model aslinya
    try:
        tokenizer = AutoTokenizer.from_pretrained(path_qwen)
    except Exception as e:
        st.toast("Meminjam tokenizer dari Qwen base...")
        tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")

    # Memuat bobot otak LoRA (adapter_config.json)
    model = AutoPeftModelForCausalLM.from_pretrained(
        path_qwen,
        device_map="auto",
        torch_dtype=torch.float16
    )
    return tokenizer, model

tokenizer, model = load_model()

# 3. Manajemen Riwayat Obrolan (Session State)
if "messages" not in st.session_state:
    st.session_state.messages = []
    st.session_state.messages.append({"role": "assistant", "content": "Halo! Saya adalah asisten kebugaranmu. Ada yang bisa saya bantu terkait program gym hari ini?"})

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# 4. Menangkap Input Pengguna & Menghasilkan Jawaban
if prompt := st.chat_input("Ketik pertanyaan gym di sini..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Menganalisis anatomi dan program latihan..."):
            teks_input = f"<|im_start|>system\nYou are a helpful, friendly, and expert fitness assistant. Answer the user's questions about gym and workouts accurately.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

            inputs = tokenizer(teks_input, return_tensors="pt").to("cuda")

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=150,
                    temperature=0.3,
                    top_p=0.9,
                    repetition_penalty=1.1,
                    pad_token_id=tokenizer.eos_token_id
                )

            full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            jawaban = full_text.split("assistant\n")[-1].strip()

            st.markdown(jawaban)
            st.session_state.messages.append({"role": "assistant", "content": jawaban})

In [ ]:
!npm install -g localtunnel
!streamlit run app.py & npx localtunnel --port 8501